# Week 3 — Closed-loop control on the GG4 Brain

This notebook is the runnable deliverable for Week 3.  It tells the end-to-end story:

1. **Install / load the Brain** (wheel from `wheels/`, then `from GG4 import Brain`).
2. **Probe** the Brain with i.i.d. `Uniform[0, 1]` inputs to collect `(Y, U)`.
3. **Fit** an affine LGSSM with EM and known inputs (`estimator_new.EstimatorNew`).
4. **Validate** the identification (EM monotonicity, ‖CB‖, degenerate-flag check, reachability zonotope).
5. **Run closed-loop** five controllers (OpenLoop, ProportionalFeedback, LQG, PolePlacement, MPC) on two objectives (suppression `r = 0`, feasible setpoint at the zonotope midpoint).
6. **Save plots** and print a narrative summary.

All math, controllers, observer, and identification live in `week3/control/` and `week3/estimator_new.py`.  Per the spec only `control/control_interface.py` imports the estimator; everything else consumes the fitted matrices `(A, B, C, Q, R, a, c)`.

**Mapping to `week3.ipynb` tasks:**

| `week3.ipynb` task | Satisfied by |
|---|---|
| 1. Define a control objective & justify | Setpoint regulation at a reachability-feasible target (see §3 / §4 of this notebook). |
| 2. Design strategy; compare; justify final | Five controllers compared in §6, with MPC chosen as final on the basis of the study in `results/`. |
| 3. Implement integrating Week-1 sim + Week-2 estimation | Simulator (Week 1) wrapped by `SimulatorPlant`; `estimator_new.py` is the affine EM rebuild of the Week-2 pipeline (the Week-2 `estimator.py` is **unchanged**, kept as a separate graded artifact). |
| 4. Test in closed loop | `closed_loop.run_closed_loop` driver on Brain and Simulator scenarios. |
| 5. Evaluate (stability / responsiveness / effort / robustness / sensitivity) | Metrics + sweeps in `experiments/m4_study.py`; numbers in `results/RESULTS.md`. |
| 6. Show working closed loop + identified issues + attempted improvement | Figures 1–10 + `fig_brain_multiseed.png` + the documented v1 → v2 estimator rebuild as the improvement. |

## 1. Install / import

Install the Brain wheel exactly as in `week3.ipynb` (one-shot, can be skipped if already installed).

In [1]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'week 1'))

import numpy as np
import matplotlib.pyplot as plt
from GG4 import Brain

# Modules built for Week 3
import estimator_new
from control.plant import BrainPlant
from control.observer import SteadyStateKalman
from control.controllers import OpenLoop, ProportionalFeedback, LQG, PolePlacement, MPC
from control.closed_loop import run_closed_loop
from control.control_interface import IdentifiedSystem
from control.reachability import steady_state_gain, zonotope_vertices, feasibility
from control.metrics import rms, control_effort, saturation_fraction, settling_time
import run_week3

print('estimator_new module:', estimator_new.__file__)

estimator_new module: C:\Github\GG4\week3\estimator_new.py


## 2. Single-seed end-to-end run

Probe (T_cal = 600) → EM fit (n = 4) → validate → 5 controllers × 2 objectives → save
`results/run_week3_seed{seed}.png`.

The function below is exactly what `python run_week3.py --seed 0` runs from the command line.

In [2]:
summary = run_week3.run_brain_story(seed=0, T_cal=600, T_run=200, n_latent=4)

  Brain seed 0  —  walking the v3 §6 story

  [1/5] SETUP — probe + EM fit + PCA readout ...
    fit iters         : 3
    ρ(A_fit)          : 0.8404
    ‖CB‖_F            : 2.4181  (threshold 0.2108)
    B col norms       : [0.433, 0.427]
    ‖a‖, ‖c‖          : 0.318, 18.686
    degenerate flags  : none
    controller warnings: none
    one-step pred RMS : 1.012  (20.7% of y-RMS)
    readout (2 PCs)   : explained var = [np.float64(0.589), np.float64(0.079)]
    cond(G_readout)   : 95.97   (ILL-CONDITIONED — inputs cannot move PC1, PC2 independently)

  [2/5] WHAT CAN WE HOLD — reachable (PC1, PC2) region from the fit ...
    G_fit (input → readout DC gain) =
[[15.27301436  1.72500234]
 [ 1.48791728  0.33078985]]
    z0_fit (resting readout)        = [10.50548401 -0.72212956]
    feasible target = G·[0.5, 0.5] + z0 = [19.004  0.187]  (zonotope midpoint)
    is ref=(0,0) holdable?            False  — suppression below the resting readout is infeasible
    is the midpoint target holdabl

## 3. Reading the suppression numbers honestly

On most Brain seeds the suppression target `r = 0` is **outside the affine reachable zonotope** — the resting readout `z0 = C(I-A)^{-1} a + c` already has magnitude > 0, and the non-negative input cannot push the system *below* that resting state.  Consequences:

- **OpenLoop** is exactly at z-RMS ≈ ‖z0‖, the suppression floor.
- **MPC** notices the infeasibility (its constraint-aware planner) and outputs `u ≡ 0`, **matching open-loop**.
- **LQG** loses ~5–10 % because its un-clipped law would request negative input, and clipping makes the closed-loop slightly worse than OL on a symmetric system.
- **ProportionalFeedback** can gain a few percent because it modulates around the bias point.

This is the honest negative result for the suppression objective on a one-sided actuator (spec §7 calls this out: *“the systems are already stable, so there is nothing to stabilise — state this explicitly”*; the corollary is that suppression below the natural floor is unreachable).

## 4. The supported objective — setpoint regulation at a feasible target

The reachability check tells you which targets are achievable.  We pick the zonotope midpoint `G·[0.5, 0.5] + z0` as the canonical feasible target.

On every seed tested (next section), LQG and MPC reach this target to within ~0.2 in each readout channel — this is the spec's *primary* objective and the one the one-sided actuator robustly supports.

## 5. Multi-seed sweep (≥5 seeds)

Re-runs the whole pipeline for the first 5 Brain seeds.  Each seed gets its own `results/run_week3_seed{N}.png`.

*(Heavy: takes a couple of minutes.  Adjust `args.all_seeds` to taste.)*

In [3]:
results = []
for s in range(5):
    results.append(run_week3.run_brain_story(seed=s, T_cal=600, T_run=200, n_latent=4))

print('\nFeasible setpoint achieved across seeds:')
for r in results:
    err = r['setpoint']['MPC']['rms_err']
    ach = r['setpoint']['MPC']['achieved']
    ref = r['ref_set']
    print(f"  seed {r['seed']}: ref={np.round(ref,3)} -> achieved={np.round(ach,3)} (rms_err={err:.3f})")

  Brain seed 0  —  walking the v3 §6 story

  [1/5] SETUP — probe + EM fit + PCA readout ...
    fit iters         : 3
    ρ(A_fit)          : 0.8404
    ‖CB‖_F            : 2.4181  (threshold 0.2108)
    B col norms       : [0.433, 0.427]
    ‖a‖, ‖c‖          : 0.318, 18.686
    degenerate flags  : none
    controller warnings: none
    one-step pred RMS : 1.012  (20.7% of y-RMS)
    readout (2 PCs)   : explained var = [np.float64(0.589), np.float64(0.079)]
    cond(G_readout)   : 95.97   (ILL-CONDITIONED — inputs cannot move PC1, PC2 independently)

  [2/5] WHAT CAN WE HOLD — reachable (PC1, PC2) region from the fit ...
    G_fit (input → readout DC gain) =
[[15.27301436  1.72500234]
 [ 1.48791728  0.33078985]]
    z0_fit (resting readout)        = [10.50548401 -0.72212956]
    feasible target = G·[0.5, 0.5] + z0 = [19.004  0.187]  (zonotope midpoint)
    is ref=(0,0) holdable?            False  — suppression below the resting readout is infeasible
    is the midpoint target holdabl

## 6. What changed since v1 — the attempted improvement

The v1 attempt used the Week-2 `estimator.py` directly: mean-centred Y, unit-norm B columns, no `a` / `c` offsets.  On the Brain it left LQG **destabilising 4 of 5 seeds (4–8× worse than open-loop)** and MPC achieving a setpoint with the wrong sign in raw output coordinates.

The fix is `estimator_new.py`: an affine LGSSM with EM, known inputs and explicit offsets `a, c`, fit in **true input units**.  This eliminates the bias-point + sign-of-`B` failure and lets the feedforward `u_ff` land inside `[0, 1]` instead of clipping to zero.

Concrete v1 → v2 deltas on the same 5 Brain seeds (numbers in `results/RESULTS.md`):

| seed | v1 LQG / OpenLoop | v2 LQG / OpenLoop |
|------|---|---|
| 0 | 1.00 (`u ≡ 0`)  | 1.10 (non-zero u) |
| 1 | 5.13 (destabilised) | 1.00 |
| 2 | 7.16 | 0.93 |
| 3 | 7.97 | 0.92 |
| 4 | 4.43 | 1.00 |

Plus: **MPC over LQG** as the secondary improvement — MPC Pareto-dominates clipped-LQR on suppression and is the only controller that does something useful on infeasible setpoints (settles at the closest feasible point).

## 7. Where to look for everything else

| What | Where |
|---|---|
| Per-milestone narrative + numbers | `results/RESULTS.md` (v2 sections at the top; v1 appendix below) |
| Figures 1–10 + Brain multi-seed | `results/*.png` |
| Estimator EM + warm start | `estimator_new.py` |
| Controllers (5) | `control/controllers.py` |
| Observer | `control/observer.py` |
| Reachability zonotope | `control/reachability.py` |
| Closed-loop driver | `control/closed_loop.py` |
| Identification boundary | `control/control_interface.py` — **only importer of `estimator_new`** |
| Unit tests | `tests/` — 10 cases, all pass |
| End-to-end script | `run_week3.py` |
